In [0]:
--the real Silver step — combining ai_parse_document + ai_classify into one query that actually saves a row into your silver_parsed_documents table


WITH parsed AS (
  SELECT
    path,
    ai_parse_document(content, MAP('version', '2.0')) AS parsed_content
  FROM READ_FILES(
    '/Volumes/employee_mgmt/doc_intelligence/raw_documents/03_Company_Handbook.pdf',
    format => 'binaryFile'
  )
)
INSERT INTO silver_parsed_documents
SELECT
  uuid()                              AS document_id,
  path                                 AS file_path,
  '03_Company_Handbook.pdf'            AS file_name,
  parsed_content,
  ai_classify(
    parsed_content,
    '{"Employee Contract": "A signed employment agreement", "HR Policy": "A company-wide HR policy document", "Offer Letter": "A pre-employment offer", "NDA": "A confidentiality agreement", "Other": "Anything else"}',
    MAP('version', '2.0')
  )                                    AS detected_type,
  'HR'                                 AS uploaded_by,
  current_timestamp()                  AS upload_ts
FROM parsed;


UPDATE silver_parsed_documents
SET detected_type = get_json_object(to_json(detected_type), '$.response[0]')
WHERE document_id = '3ecb29ad-18d1-46ab-b4ad-f01e72b66854';



UPDATE silver_parsed_documents
SET detected_type = get_json_object(detected_type, '$.response[0]')
WHERE document_id = '3ecb29ad-18d1-46ab-b4ad-f01e72b66854';